# Постановка эксперимента и запуск

## Что проверяю

Задание просит проверить EdgeMTSC на своих многомерных временных данных и сравнить качество, параметры и время.

Я делаю чуть больше:

1. standard EdgeMTSC vs Transformer -- основной baseline;
2. standard EdgeMTSC vs моя pairwise-модификация;
3. внутри EdgeMTSC короткая ablation `w/o IMP` и `single-scale LKB`, чтобы отдельно проверить channel relations и разные temporal scales. Эту часть трактую как вспомогательную: она сделана на одном subject fold.

## Почему UCI HAR

Нужен был корпус, который одновременно:

- действительно многомерный;
- достаточно большой для содержательного DL-эксперимента;
- обучаю все локально на ноуте, поэтому не беру слишком тяжелый корпус;
- имеет физически осмысленные связанные каналы;
- популярен и имеет внешние baseline;
- имеет корректный official split.

UCI HAR попадает ровно в этот баланс: 10299 окон, 9 каналов, 128 шагов, 6 классов, 30 участников. Official train/test разделены по людям.

Источник на Hugging Face: https://huggingface.co/datasets/udayl/UCI_HAR

## Как запускать

Из корня репозитория:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

После этого открыть VS Code / Jupyter и запускать ноутбуки строго по порядку:

```text
00_EDA.ipynb
01_EdgeMTSC.ipynb
02_EdgeMTSC_pairwise.ipynb
03_Transformer.ipynb
```

Первый ноутбук скачивает данные, поэтому остальные предполагают, что `00_EDA.ipynb` уже выполнен.

Запускаю на MacBook Pro с M4 Pro и 24 GB unified memory. PyTorch автоматически использует `mps`; в каждом model notebook device печатается первой ячейкой.

## Протокол

- seed = 42;
- batch size = 128;
- AdamW;
- lr = 8e-4;
- weight decay = 1e-3;
- label smoothing = 0.05;
- максимум 35 эпох на CV fold;
- early stopping patience = 7;
- 3-fold GroupKFold по subject;
- final epochs = median(best epoch по CV);
- z-score fit только на train-части соответствующего fold;
- official test не участвует в preprocessing, CV, early stopping и выборе числа эпох; финальные метрики считаю уже для зафиксированных конфигураций.

## Метрики

`Accuracy` -- headline-метрика для сопоставления с опубликованными HAR baseline. Для early stopping и выбора модели использую `macro F1`.

Дополнительно считаю:

- balanced accuracy;
- macro F1;
- per-class F1;
- confusion matrix;
- train time;
- inference latency per sample;
- trainable parameter count.

Так видно не только общий score, но и цену модели и то, на каких классах она ошибается.
